In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
results = {'prefix':[], 'correct':[], 'partial':[], 'incorrect':[], 'irrelevant':[]}
for file in os.listdir('output'):
    if file.startswith('eval_') and file.endswith('.jsonl'):
        correct, partial, incorrect, irrelevant = 0, 0, 0, 0
        with open(os.path.join('output', file), 'r') as f:
            data = [json.loads(line) for line in f.readlines()]
            for item in data:
                if item['evaluation'].find('Correct') != -1 and item['evaluation'].find('Partially Correct') == -1:
                    correct += 1
                elif item['evaluation'].find('Partially Correct') != -1:
                    partial += 1
                elif item['evaluation'].find('Incorrect') != -1:
                    incorrect += 1
                elif item['evaluation'].find('Irrelevant') != -1:
                    irrelevant += 1
                else:
                    raise ValueError(f"Unknown evaluation: {item['evaluation']}")
        results['prefix'].append(file[len('eval_medgemma_'):-len('.jsonl')])
        results['correct'].append(correct)
        results['partial'].append(partial)
        results['incorrect'].append(incorrect)
        results['irrelevant'].append(irrelevant)
        
df = pd.DataFrame(results)

In [3]:
df

,prefix,correct,partial,incorrect,irrelevant
0,main_clinician_neutral,550,151,339,21
1,main_clinician_anger_frustration,549,162,328,22
2,main_clinician_sadness_distress,546,168,316,31
3,main_patient_anger_frustration,555,168,309,29
4,main_clinician_fear_anxiety,556,169,308,28
5,main_patient_neutral,547,167,316,31
6,default,550,145,347,19
7,main_patient_sadness_distress,560,167,307,27
8,main_patient_fear_anxiety,559,162,303,37


In [10]:
import json
import pandas as pd

def compare_eval_files(file1, file2):
    with open(file1, 'r', encoding='utf-8') as f:
        data1 = [json.loads(line) for line in f]

    with open(file2, 'r', encoding='utf-8') as f:
        data2 = [json.loads(line) for line in f]

    diffs = []
    for i, (item1, item2) in enumerate(zip(data1, data2)):
        eval1 = item1.get('evaluation')
        eval2 = item2.get('evaluation')

        if eval1 != eval2:
            diffs.append({
                'index': i,
                'evaluation_file1': eval1,
                'evaluation_file2': eval2,
                'gt': item1.get('answer'),
                'question_file1': item1.get('question'),
                'question_file2': item2.get('question'),
                'answer_file1': item1.get('model_answer', item1.get('answer')),
                'answer_file2': item2.get('model_answer', item2.get('answer')),
            })

    return pd.DataFrame(diffs)

In [11]:
dff = compare_eval_files('output/eval_medgemma_default.jsonl', 'output/eval_medgemma_main_patient_neutral.jsonl')

In [15]:
dff[dff['evaluation_file1'].isin(['Correct','Incorrect']) & dff['evaluation_file2'].isin(['Correct','Incorrect'])]

,index,evaluation_file1,evaluation_file2,gt,question_file1,question_file2,answer_file1,answer_file2
2,15,Incorrect,Correct,No,Does the picture contain liver?,Does the picture contain liver?,"No, the picture contains the chest and upper a...","Based on the image provided, the liver is not ..."
4,24,Correct,Incorrect,Lung,What is the largest organ in the picture?,What is the largest organ in the picture?,The largest organ in the picture is the lung.,The largest organ visible in the picture is th...
6,33,Correct,Incorrect,Lung,What is the largest organ in the picture?,What is the largest organ in the picture?,The largest organ in the picture is the lung.,"Based on the image, the largest organ visible ..."
9,45,Incorrect,Correct,Yes,Does the picture contain spleen?,Does the picture contain spleen?,"No, the picture contains the liver, not the sp...","Based on the image, the spleen is visible on t..."
10,49,Correct,Incorrect,Lung,What is the largest organ in the picture?,What is the largest organ in the picture?,The largest organ in the picture is the lung.,"Based on the image, the largest organ visible ..."
...,...,...,...,...,...,...,...,...
160,973,Correct,Incorrect,Rectum,Which organs/organ in the picture belong to th...,Which organs/organ in the picture belong to th...,The rectum is part of the digestive system.,"Based on the image, the **bladder** is an orga..."
161,983,Correct,Incorrect,Store urine,What is the effect of the gray organ on the ce...,What is the effect of the gray organ on the ce...,The gray organ in the center of the picture is...,"Based on the image, the gray organ in the cent..."
163,1007,Incorrect,Correct,Yes,Is there spinal cord in this picture?,Is there spinal cord in this picture?,"Based on the provided image, there is no visib...","Based on the image, the spinal cord is visible..."
164,1023,Correct,Incorrect,Brain,What is the largest organ in the picture?,What is the largest organ in the picture?,The largest organ in the picture is the brain.,"Based on the image, the largest organ is the b..."


In [14]:
dff.iloc[4]['answer_file1'], dff.iloc[4]['answer_file2']

('The largest organ in the picture is the lung.',
 'The largest organ visible in the picture is the lung.')

In [8]:
dff = compare_eval_files('output/eval_medgemma_default.jsonl', 'output/eval_medgemma_main_patient_neutral.jsonl')

Need to compare: 

- main_clinician_neutral vs default
- main_patient_neutral vs default
- main_clinician_neutral vs main_patient_neutral

- And then, for each emotion, compare vs neutral condition